# Using QCrBoxAPIClient to interact with QCrBox

In addition to the API endpoints available at http://127.0.0.1:11000/api (or whatever the the production URL is), we
also offer a Python API client which simplifies the serialisation of requests/responses to/from the QCrBox API using 
Python dataclasses.

The API client can installed by adding it to your `poetry` project, if you use poetry, using 
`poetry add git+https://github.com/QCrBox/QCrBoxAPIClient`. It can also be installed using `pip`. The client only
requires a very minimal set of dependencies, which will be installed automatically.

In [18]:
%pip install --upgrade pip > /dev/null 2>&1

Note: you may need to restart the kernel to use updated packages.


In [19]:
%pip install git+https://github.com/QCrBox/QCrBoxAPIClient@v0.2.2 2>&1 | tail -n 1

Note: you may need to restart the kernel to use updated packages.


Invoking an interactive olex2 session.

This is an example script of creating a dataset and passing the data file within
the dataset to the API to invoke an Olex2 session accessible at:

    http://127.0.0.1:12004/vnc.html?path=vnc&autoconnect=true&resize=remote&reconnect=true&show_dot=true

NB: you will need to remove the calls to `close_interactive_session` and
`delete_dataset_by_id` to actually interact with the interactive session.


In [20]:
from qcrboxapiclient import Client

# Create a synchronous client, which is passed to each `sync()` for an API endpoint
# via keyword
client = Client(base_url="http://127.0.0.1:11000")

In [21]:
# Example file in QCrBox repository, but could substitite with `work.cif` which
# 100% works w/ Olex2 and Crystal Explorer

import pathlib

test_file = pathlib.Path("../../pyqcrbox/robot_tests/test_data/robot_test_cif.cif").resolve()


In [22]:
import io
from qcrboxapiclient.types import File
from qcrboxapiclient.models import CreateDatasetBody

# To upload the file to the registry, we need to create a dataset and attach the
# file to it. Files are sent via the API as binary in a CreateDatasetBody payload
with test_file.open("rb") as f:
    file = File(io.BytesIO(f.read()), test_file.name)
upload_payload = CreateDatasetBody(file)

In [23]:
# Uploading the file with this endpoint creates a dataset containing this file.
# Assuming everything went OK, thn we will get a QCrBoxResponse. If an error
# occurred then the response is of type QCrBoxErrorResponse

from qcrboxapiclient.api.datasets import create_dataset
from qcrboxapiclient.models import QCrBoxErrorResponse

response = create_dataset.sync(client=client, body=upload_payload)
if isinstance(response, QCrBoxErrorResponse):
    raise TypeError("Failed to upload file")
else:
    print("Created dataset:", response)

Created dataset: QCrBoxResponseDatasetsResponse(status='success', message="Created dataset: 'qcrbox_ds_0x9b22faf9ba634bb6886ae82fc9f03f20'", timestamp='2025-07-15T10:30:14.444579+00:00Z', payload=DatasetsResponse(datasets=[DatasetResponse(qcrbox_dataset_id='qcrbox_ds_0x9b22faf9ba634bb6886ae82fc9f03f20', data_files=DatasetResponseDataFiles(additional_properties={'robot_test_cif.cif': DataFileMetadataResponse(qcrbox_file_id='qcrbox_df_0x9ff1aec300ca4df5988fb5db410df0cf', filename='robot_test_cif.cif', filetype='cif', additional_properties={})}), additional_properties={})], additional_properties={}), additional_properties={})


In [24]:
# The response returns the created object in payload.datasets[0]. Note that this
# doesn't contain any of the file's binary data and instead contains metadata
# about the dataset and data files in the data set.
dataset_id = response.payload.datasets[0].qcrbox_dataset_id
data_file_id = response.payload.datasets[0].data_files[test_file.name].qcrbox_file_id

In [25]:
# To create an interactive olex2 session we call the create_interactive_session_with_arguments
# endpoint with a payload of application we want to start and arguments for the olex2
# command. For olex2, we need an argument named "input_file" which contains a data_file_id.
# Note that the arguments is a dict and we use `CreateInteractiveSessionArguments` to
# marshal out input into Json for the API

from qcrboxapiclient.api.interactive_sessions import (
    create_interactive_session_with_arguments,
)
from qcrboxapiclient.models import (
    CreateInteractiveSession,
    CreateInteractiveSessionArguments,
)

arguments = CreateInteractiveSessionArguments.from_dict({"input_file": {"data_file_id": data_file_id}})
create_session = CreateInteractiveSession("olex2", "1.5-alpha", arguments)
response = create_interactive_session_with_arguments.sync(client=client, body=create_session)
if isinstance(response, QCrBoxErrorResponse):
    raise TypeError("Failed to start interactive session")
else:
    print("Created interactive session:", response)

Created interactive session: QCrBoxResponseInteractiveSessionIDResponse(status='success', message="Command invocation accepted: 'olex2'-'1.5-alpha'", timestamp='2025-07-15T10:30:14.497434+00:00Z', payload=InteractiveSessionIDResponse(interactive_session_id='qcrbox_calc_0x4dee51f8826c4104b1ded3609c34b2f5', additional_properties={}), additional_properties={})


In [26]:
# The response from this endpoint is slightly different as it returns a reference
# to the create object, rather than the object iself. The payload contains the
# interactive_session_id which is also the calcualtion id of the interactive session
session_id = response.payload.interactive_session_id
print("Session ID:", session_id)

Session ID: qcrbox_calc_0x4dee51f8826c4104b1ded3609c34b2f5


In [27]:
# We can close the interactive session now. But we should sleep for a little while
# so things have time to registry in all the databases and the run command can
# execute

import time
from qcrboxapiclient.api.interactive_sessions import close_interactive_session

time.sleep(3)
response = close_interactive_session.sync(client=client, id=session_id)
if isinstance(response, QCrBoxErrorResponse):
    raise TypeError("Failed to close interactive session")
else:
    print("Closed interactive session:", response)

Closed interactive session: QCrBoxResponseInteractiveSessionClosedResponse(status='success', message="Closed interactive session: 'qcrbox_calc_0x4dee51f8826c4104b1ded3609c34b2f5'", timestamp='2025-07-15T10:30:17.588184+00:00Z', payload=InteractiveSessionClosedResponse(interactive_sessions=[CloseInteractiveSessionResponseNATS(session_id='qcrbox_calc_0x4dee51f8826c4104b1ded3609c34b2f5', status='successful', output_dataset_id='qcrbox_ds_0xd6b2423cb8a341a0953b4e2156802f37', error_msg='', additional_properties={})], additional_properties={}), additional_properties={})


In [28]:
# Delete the dataset afterward, because we are just using this for test purposes
from qcrboxapiclient.api.datasets import create_dataset, delete_dataset_by_id

response = delete_dataset_by_id.sync(id=dataset_id, client=client)
if isinstance(response, QCrBoxErrorResponse):
    raise TypeError("Failed to delete test dataset")
else:
    print("Deleted test dataset")

Deleted test dataset
